# Losses, optimization and backpropagation · Modules 7–8 and 10
Python 3 and NumPy only. Derive the equations in Lessons 27 and 38 first. The nonlinear task is noisy XOR; it makes the limitation of a linear decision boundary visible. It is a controlled teaching problem, not a real-image benchmark.


In [1]:
import numpy as np
np.set_printoptions(precision=5,suppress=True)
rng=np.random.default_rng(207)
def probabilities(z):
    shifted=z-z.max(axis=1,keepdims=True)
    exp=np.exp(shifted)
    return exp/exp.sum(axis=1,keepdims=True)
def cross_entropy(z,y):
    shifted=z-z.max(axis=1,keepdims=True)
    return np.mean(np.log(np.exp(shifted).sum(axis=1))-shifted[np.arange(len(y)),y])
print('stable probabilities:',probabilities(np.array([[1000.,1001.]])))
assert np.allclose(probabilities(np.array([[1000.,1001.]])),probabilities(np.array([[0.,1.]])))
for alpha in [.1,.75,1.,1.2]:
    w=0.;history=[]
    for step in range(8):
        history.append((w,(w-3)**2));w-=alpha*2*(w-3)
    print('learning rate:',alpha,'first/last weight,loss:',history[0],history[-1])


stable probabilities: [[0.26894 0.73106]]
learning rate: 0.1 first/last weight,loss: (0.0, 9.0) (2.3708544, 0.39582418599936026)
learning rate: 0.75 first/last weight,loss: (0.0, 9.0) (3.0234375, 0.00054931640625)
learning rate: 1.0 first/last weight,loss: (0.0, 9.0) (6.0, 9.0)
learning rate: 1.2 first/last weight,loss: (0.0, 9.0) (34.62405119999999, 1000.0806143002208)


## Split once and standardize on training only
The declared comparison uses 1200 full-batch updates at learning rate 0.15 for each model. Keep the test split until the training and validation procedure is fixed.


In [2]:
bits=rng.integers(0,2,(400,2));labels=(bits[:,0]!=bits[:,1]).astype(int)
features=bits+rng.normal(0,.06,bits.shape)
Xtr,Xval,Xtest=features[:240],features[240:320],features[320:]
ytr,yval,ytest=labels[:240],labels[240:320],labels[320:]
mu=Xtr.mean(axis=0);scale=Xtr.std(axis=0)
Xtr=(Xtr-mu)/scale;Xval=(Xval-mu)/scale;Xtest=(Xtest-mu)/scale
Y=np.eye(2)[ytr]
linear_W=np.zeros((2,2));linear_b=np.zeros(2)
for _ in range(1200):
    logits=Xtr @ linear_W+linear_b
    G=(probabilities(logits)-Y)/len(ytr)
    dW=Xtr.T @ G;db=G.sum(axis=0)
    linear_W-=.15*dW;linear_b-=.15*db
print('linear training loss:',cross_entropy(Xtr @ linear_W+linear_b,ytr))
print('linear validation accuracy:',np.mean((Xval @ linear_W+linear_b).argmax(axis=1)==yval))


linear training loss: 0.6912392225486521
linear validation accuracy: 0.5


## Implement every backward equation
The gradients are all computed before any parameter update. ReLU at exactly zero uses derivative zero by convention. The loss is a mean across examples.


In [3]:
parameters={'W1':rng.normal(0,.7,(2,12)),'b1':np.full(12,.2),
            'W2':rng.normal(0,.2,(12,2)),'b2':np.zeros(2)}
def forward(X,p):
    Z1=X @ p['W1']+p['b1'];A1=np.maximum(0,Z1)
    logits=A1 @ p['W2']+p['b2']
    return Z1,A1,logits
def loss_grad(X,y,p):
    Z1,A1,logits=forward(X,p)
    G2=(probabilities(logits)-np.eye(2)[y])/len(y)
    G1=(G2 @ p['W2'].T)*(Z1>0)
    gradients={'W2':A1.T @ G2,'b2':G2.sum(axis=0),
               'W1':X.T @ G1,'b1':G1.sum(axis=0)}
    return cross_entropy(logits,y),gradients
batch=Xtr[:8];target=ytr[:8]
_,analytic=loss_grad(batch,target,parameters)
checks=[]
for name,index in [('W1',(0,0)),('W1',(1,4)),('b1',(2,)),('W2',(3,1)),('b2',(0,))]:
    value=parameters[name][index];h=1e-5
    parameters[name][index]=value+h;plus=loss_grad(batch,target,parameters)[0]
    parameters[name][index]=value-h;minus=loss_grad(batch,target,parameters)[0]
    parameters[name][index]=value
    numeric=(plus-minus)/(2*h);error=abs(numeric-analytic[name][index])
    checks.append(error);print('gradient check:',name,index,error)
assert max(checks)<1e-6


gradient check: W1 (0, 0) 1.5176887524503968e-12
gradient check: W1 (1, 4) 6.6749383798025974e-12
gradient check: b1 (2,) 5.593925930108545e-12
gradient check: W2 (3, 1) 3.8089253973083714e-12
gradient check: b2 (0,) 3.504030399170688e-12


## Train after the derivative checks
Record both training and validation evidence. Model capacity and training quality are different questions.


In [4]:
initial=loss_grad(Xtr,ytr,parameters)[0];history=[]
for step in range(1200):
    loss,gradients=loss_grad(Xtr,ytr,parameters)
    for name in parameters: parameters[name]-=.15*gradients[name]
    if step%200==0: history.append((step,loss))
final=loss_grad(Xtr,ytr,parameters)[0]
validation=forward(Xval,parameters)[2]
assert np.isfinite(final) and final<initial
print('MLP loss history:',history)
print('MLP final loss:',final,'validation accuracy:',np.mean(validation.argmax(axis=1)==yval))
# The procedure is fixed; inspect held-out test outcomes now.
linear_test=(Xtest @ linear_W+linear_b).argmax(axis=1)
mlp_logits=forward(Xtest,parameters)[2];mlp_test=mlp_logits.argmax(axis=1)
print('test count:',len(ytest),'linear accuracy:',np.mean(linear_test==ytest),
      'MLP accuracy:',np.mean(mlp_test==ytest),'MLP NLL:',cross_entropy(mlp_logits,ytest))
confusion=np.zeros((2,2),dtype=int)
np.add.at(confusion,(ytest,mlp_test),1)
print('confusion rows=true, columns=predicted:\n',confusion)


MLP loss history: [(0, np.float64(0.6722901032251598)), (200, np.float64(0.013257432994269886)), (400, np.float64(0.004934506977554206)), (600, np.float64(0.0028704866435060043)), (800, np.float64(0.0019831279498043705)), (1000, np.float64(0.00149865758190865))]
MLP final loss: 0.001196394561183955 validation accuracy: 1.0
test count: 80 linear accuracy: 0.55 MLP accuracy: 1.0 MLP NLL: 0.0010281993217461669
confusion rows=true, columns=predicted:
 [[36  0]
 [ 0 44]]


## Your experiment
Write the forward and backward shapes without notes. Verify at least two new parameter coordinates. Declare one comparison using a fresh held-out split: remove the ReLU, change learning rate, or add a squared penalty with an explicitly stated reduction. Change one factor at a time. Report unsuccessful settings too, and explain why this synthetic result does not establish real-image performance.
